In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler, MinMaxScaler
import warnings

warnings.filterwarnings('ignore')

In [ ]:
def setup_directories(base_dir="NASNetLarge"):
    plots_dir = os.path.join(base_dir, "plots")
    files_dir = os.path.join(base_dir, "files")

    for directory in [base_dir, plots_dir, files_dir]:
        os.makedirs(directory, exist_ok=True)

    return base_dir, plots_dir, files_dir

In [ ]:
def load_data(file_path):
    print(f"Loading data from {file_path}")
    df = pd.read_csv(file_path)

    num_features = 2024

    # Validate data structure
    if len(df.columns) < num_features + 3:
        print(f"Warning: Expected at least {num_features + 3} columns, found {len(df.columns)}")
        num_features = len(df.columns) - 3  # Adjust to use available feature columns

    feature_columns = df.columns[:num_features].tolist()
    meta_columns = df.columns[num_features:].tolist()

    print(f"Feature columns: {len(feature_columns)}")
    print(f"Metadata columns: {meta_columns}")

    return df, feature_columns, meta_columns

In [ ]:
def apply_standardization(df, feature_columns, method="robust"):
    df_standardized = df.copy()

    if method == "zscore":
        # Standard Z-score with outlier handling
        for col in feature_columns:
            values = df[col].values
            mean = np.mean(values)
            std = np.std(values)

            if std == 0 or np.isnan(std):
                df_standardized[col] = 0
            else:
                # Apply Z-score with outlier capping
                z_scores = (values - mean) / std
                z_scores = np.clip(z_scores, -3, 3)  # Cap at ±3 standard deviations
                df_standardized[col] = z_scores

    elif method == "robust":
        # Use RobustScaler which is less sensitive to outliers
        scaler = RobustScaler(quantile_range=(25, 75))
        df_standardized[feature_columns] = scaler.fit_transform(df[feature_columns])

    elif method == "minmax":
        # MinMaxScaler to get values in a specific range
        scaler = MinMaxScaler(feature_range=(-1, 1))
        df_standardized[feature_columns] = scaler.fit_transform(df[feature_columns])

    else:
        raise ValueError(f"Unknown standardization method: {method}")

    return df_standardized

In [ ]:
base_dir, plots_dir, files_dir = setup_directories()
file_path = "../2_Feature_Extraction/NASNetLarge/feature_extract_Hybrid_features.csv"

In [ ]:
df, feature_columns, meta_columns = load_data(file_path)

print("\nOriginal data summary (first 5 features):")
print(df[feature_columns[:5]].describe().round(4))

In [ ]:
 # Choose standardization method
print("\nChoose standardization method:")
print("1: Robust Scaling (recommended for consistent distribution)")
print("2: Z-score with outlier capping")
print("3: Min-Max scaling to range [-1, 1]")

choice = 2

if choice == "2":
    method = "zscore"
    print("\nApplying Z-score standardization with outlier capping...")
elif choice == "3":
    method = "minmax"
    print("\nApplying Min-Max scaling to range [-1, 1]...")
else:
    method = "robust"
    print("\nApplying Robust Scaling (based on median and IQR)...")

In [ ]:
# Apply standardization
standardized_df = apply_standardization(df, feature_columns, method=method)

print("\nStandardized data summary (first 5 features):")
print(standardized_df[feature_columns[:5]].describe().round(4))

In [ ]:
# Save the standardized data to CSV
output_file = os.path.join(files_dir, f'standardized_data_{method}.csv')
standardized_df.to_csv(output_file, index=False)
print(f"\nStandardized data saved to {output_file}")

In [ ]:
max_features = 20
original_df = df

In [ ]:
# Select features for visualization
if len(feature_columns) > max_features:
    non_zero_counts = {col: np.count_nonzero(original_df[col]) for col in feature_columns}
    selected_features = sorted(non_zero_counts.keys(),
                               key=lambda x: non_zero_counts[x],
                               reverse=True)[:max_features]
    print(f"Selected {max_features} features with most non-zero values for visualization")
else:
    selected_features = feature_columns

# Create rainbow color palette
colors = plt.cm.rainbow(np.linspace(0, 1, len(selected_features)))

In [ ]:
# --- BEFORE STANDARDIZATION PLOT ---
plt.figure(figsize=(16, 8))
sns.set_style("whitegrid")

plt.rcParams['font.family'] = 'Times New Roman'

plot_data = pd.DataFrame()
for i, feature in enumerate(selected_features):
    plot_data[feature] = original_df[feature]

plot_data_melted = pd.melt(plot_data, var_name='Feature', value_name='Value')
sns.boxplot(x='Feature', y='Value', data=plot_data_melted,
            palette=colors, showfliers=False, width=0.6)

plt.grid(False)

plt.title('Before Z-score', fontsize=28, pad=20)
plt.ylabel('Values', fontsize=24)
plt.xlabel('Features', fontsize=24)
plt.xticks(fontsize=18)
plt.yticks(fontsize=18)

plt.tight_layout()
plt.savefig(os.path.join(plots_dir, 'before_standardization.png'), dpi=1000)
plt.savefig(os.path.join(plots_dir, 'before_standardization.pdf'), dpi=1000)
plt.show()
plt.close()

In [ ]:
# --- AFTER STANDARDIZATION PLOT ---
plt.figure(figsize=(16, 8))
plt.rcParams['font.family'] = 'Times New Roman'
std_plot_data = pd.DataFrame()

for i, feature in enumerate(selected_features):
    std_plot_data[feature] = standardized_df[feature]

std_plot_data_melted = pd.melt(std_plot_data, var_name='Feature', value_name='Value')

ax = sns.boxplot(x='Feature', y='Value', data=std_plot_data_melted,
                 palette=colors, showfliers=False, width=0.6,
                 whiskerprops={'linewidth': 1.5},
                 capprops={'linewidth': 1.5})
ax.grid(False)

# Customize font sizes
plt.title('After Z-score', fontsize=28, pad=20)
plt.ylabel('Values', fontsize=24)
plt.xlabel('Features', fontsize=24)
ax.tick_params(axis='y', labelsize=18) 
ax.tick_params(axis='x', labelsize=18)

plt.tight_layout()
plt.savefig(os.path.join(plots_dir, 'after_standardization.png'), dpi=1000)
plt.savefig(os.path.join(plots_dir, 'after_standardization.pdf'), dpi=1000)
plt.show()
plt.close()

In [ ]:
print("\nComplete standardization and visualization process")